# Spatial Discretization
**강좌**: *전산유체역학*

## 공간차분 기법
### 개요
- Semi discrtet form

    $$
    \frac{d\mathbf{U}_{i,j}}{dt} = \mathbf{R}_{i,j} = -J_{i,j} (\tilde{\mathbf{F}}_{i+1/2,j} - \tilde{\mathbf{F}}_{i-1/2,j} + \tilde{\mathbf{G}}_{i,j+1/2} - \tilde{\mathbf{G}}_{i,j-1/2})
    $$

- Riemann 문제 해석
    * 수치적으로 근사 Riemann 해석을 위한 Flux 기법을 적용한다.

        $$
        \tilde{\mathbf{F}}_{i+1/2,j} = \mathbf{H}(\mathbf{U}^L_{i+1/2,j}, \mathbf{U}^R_{i+1/2,j}, \mathbf{n})
        $$


        * $n$ 은 Face metric인 $|\nabla \xi| / J$ 또는 $|\nabla \eta| / J$을 사용한다.
     
    * 여기서 $\mathbf{U}_{i+1/2,j}^L, \mathbf{U}_{i+1/2,j}^R$ 은 인접 셀 값으로 1차 정확도 기법은 평균값을 사용한다.
    
        $$
        \mathbf{U}_{i+1/2,j}^L = \mathbf{U}_{i,j}, \mathbf{U}_{i+1/2,j}^R = \mathbf{U}_{i+1,j}
        $$
 
    * 다양한 근사 리만 해석 기법 (Flux schemes) 적용 가능함
    
- High Resolution 기법은 셀 내부의 분포를 고려하지만 충격파 부근에서 수치 안정성을 확보할 수 있는 Shock-capturing 기법이 필수적이다.
 
- 경계 조건

    - 경계 조건의 형태가 다양해짐
 
        - 다양한 입구 또는 출구 조건 적용 (Todo)
 
    - 위 / 아랫면 (jmin / jmax) 또는 양 끝면 (imin, imax) 에 대해 대칭 조건도 가능함 
    
### TVD-MUSCL 기법    
- 인접 셀 정보로 부터 기울기를 재구성한다.
    - 벡터의 컴포넌트별 $u$ 로 적용함
    - 보존벡터 (Conservative Variable), 원시벡터 (Primitive Variable) 또는 고유벡터 (Eigenvector)의 컴포넌트 별로 적용 가능

    $$
    \begin{align}
    u^{L}_{i+1/2,j} &= u_{i,j} + 0.5 \phi(r_i) \Delta u_{i+1/2,j} \\
    u^{R}_{i+1/2,j} &= u_{i+1,j} - 0.5 \phi(r_{i+1}) \Delta u_{i+3/2,j}
    \end{align}
    $$

    여기서

    $$
    \Delta u_{i+1/2,j} = u_{i+1,j} - u_{i,j}
    $$
    
- $\phi(r)$ 은 기울기 제한자이다. TVD 조건을 만족하는 아래와 같이 대표적인 기법들이 있다.

   * minmod : $\phi(r) = \max(0, \min(1, r))$
   * Superbee : $\phi(r) = \max(0, \min(2r, 1), \min(r, 2))$
   * Van Leer : $\phi(r) = \frac{r + |r|}{1+ |r|}$
   
    $$
    r_i = \frac{\Delta u_{i-1/2,j}}{\Delta u_{i+1/2,j}}
    $$
    
    * $r_i$ 계산시 Uniform flow 영역에서 분모가 0이 되는 것을 방지하기 위해 분모 분자 모두에 매우 작은 수 $\epsilon\approx 10^{-12}$ 수를 더한다.
    
- $\eta$ 방향에 대해서도 적용 가능함.

    - $\Delta u_{i,j+1/2} = u_{i,j+1} - u_{i,j}$

## 구현
### TVD-MUSCL 기법 추가
- TVD Limiter 함수

In [1]:
import numba as nb


# Very small number
eps = 1e-10


def get_limiter(method):
    """
    기울기 제한자 함수 생성
    
    Parameter
    ---------
    method : string
        이름
        
    Return
    ------
    limiter : jitted function
        Limiter 함수
    """
    if method == 'minmod':
        print('Minmod limiter')
        f = lambda r: max(0.0, min(r, 1.0))
    elif method == 'superbee':
        print('Superbee limiter')
        f = lambda r: max(0.0, min(2.0*r, 1.0),min(r, 2.0))
    elif method == 'vanleer':
        print('Van Leer limiter')
        f = lambda r: (r + abs(r)) / (1.0 + abs(r))
    elif method == 'unlimited':
        print('Unlimited second-order')
        f = lambda r: 1.0
    else:
        print("First-order")
        f = lambda r : 0.0
        
    f_jitted = nb.jit(f, nopython=True)
    
    @nb.jit(nopython=True)
    def limiter(dp, dm):
        if abs(dm) < eps:
            return 1.0
        else:
            return f_jitted(dp/dm)
    
    return limiter


- MUSCL 재구성시 경계 전후로 양쪽에서 2개씩 격자 정보를 요구한다. 경계에서는 다음 두 방법 모두 사용 가능하다.
    * 경계에서만 1차 정확도로 계산
    * Padding 개수를 늘림

- `rhside` 커널 재구성
    - TVD-MUSCL 재구성 포함
    - MUSCL Reconstruction 계산시 양끝 경계에서 추가적인 인접 격자가 필요함
        - ex) $1/2$ 경계에서 $\Delta u_{1/2}, \Delta u_{-1/2}$ 이 필요함
        - 일반적인 BC: 경계면에서 1차 정확도로만 계산
        - Periodic BC 등: Ghost Cell을 늘림
    - 조건에 따라 Riemann Solver, BC 커널을 생성하는 `get_rsolver`, `make_bc` 함수를 이용
        - 아래 구현함
    - `solvers.rhside.py` 에 저장

### 실습

### Riemann Solver 추가
- 여러 Flux 함수를 선택할 수 있도록 `solvers.rsolvers` 내 `__init__.py` 를 다음과 같이 작성한다.

    - Flux 기법을 추가할 경우 그 모듈을 불러오는 구문도 같이 추가한다.

In [2]:
%%writefile solvers/rsolvers/__init__.py
from solvers.fluid import to_flux

# Import implemented schemes!!!
from solvers.rsolvers.roe import make_roe
from solvers.rsolvers.rusanov import make_rusanov
from solvers.rsolvers.hll import make_hllc
from solvers.rsolvers.ausm import make_ausmp


import re


def get_rsolver(name, *args, **kwargs):
    """
    Ge Rsolvers
    ------------
    
    Parameters
    ----------
    name : string
        name of scheme
        
    Return
    ------
    flux : jitted function
        kernel from 'make_(name)' generator
    """
    print('Riemann Solver : {}'.format(name))
    
    # replace + in name with 'p' (ex. ausm+ -> ausmp)
    fname = re.sub(r'\+', 'p', name)
    
    # Run the kernal generator make_(name)
    flux = eval('make_' + fname)(*args, **kwargs)
    
    # Return jitted kernel
    return flux

Overwriting solvers/rsolvers/__init__.py


### Roe's FDS
Roe's FDS 는 다음과 같이 확장할 수 있다.

$$
\mathbf{H}(\mathbf{U}_L, \mathbf{U}_R, \mathbf{n}) 
= \frac{\mathbf{F}_{n,L} + \mathbf{F}_{n,R}}{2} - \frac{1}{2} |\hat{\mathbf{A}}| \Delta \mathbf{U}
= \frac{\mathbf{F}_{n,L} + \mathbf{F}_{n,R}}{2} - \frac{1}{2} \sum_{i=1}^m \hat{\alpha}_i |\hat{\lambda}_i| \hat{\mathbf{v}}_{i}
$$

여기서 $\hat{\mathbf{A}} = \hat{\mathbf{B}}\hat{\mathbf{C}}^{-1}$ 이고, $\Delta \mathbf{U} = \hat{\mathbf{B}} \Delta \mathbf{Q}$, $\Delta \mathbf{F} = \hat{\mathbf{C}} \Delta \mathbf{Q}$ 이다. 이때 Parameter vector $\mathbf{Q}$ 다음과 같이 도입한다.

$$
\mathbf{Q} = \sqrt{\rho} \begin{bmatrix}
1 \\ u \\ v \\ h_t
\end{bmatrix}
$$

이를 이용해서 구한 $\mathbf{A}$ 의 고유치와 고유벡터는 다음과 같다.

$$
\lambda_{1,2} = \hat{U}_n,
\lambda_3 = \hat{U}_n - \hat{a}, 
\lambda_4 = \hat{U}_n + \hat{a}.
$$

$$
\hat{\mathbf{v}}_{1} = \begin{bmatrix}
1 \\ \hat{u} \\ \hat{v} \\ \frac{\hat{u}^2 + \hat{v}^2}{2} 
\end{bmatrix}, 
\hat{\mathbf{v}}_{2} = \begin{bmatrix}
0 \\ n_y \\ -n_x \\ \hat{u} n_y - \hat{v} n_x
\end{bmatrix},
\hat{\mathbf{v}}_{3} = \begin{bmatrix}
1 \\ \hat{u} - n_x \hat{a} \\ \hat{v} - n_y \hat{a} \\ h_t - \hat{U}\hat{a}
\end{bmatrix}, 
\hat{\mathbf{v}}_{4} = \begin{bmatrix}
1 \\ \hat{u} + n_x \hat{a} \\ \hat{v} + n_y \hat{a} \\ h_t + \hat{U}\hat{a}
\end{bmatrix}
$$

여기서 Roe average variable $\hat{()}$ 는 다음과 같다.

$$
\begin{align}
\hat{\rho} & = \sqrt{\rho_L \rho_R} \\
\hat{u} &= \frac{\sqrt{\rho_L} u_L + \sqrt{\rho_R} u_R}{\sqrt{\rho_L} + \sqrt{\rho_R}} \\
\hat{v} &= \frac{\sqrt{\rho_L} v_L + \sqrt{\rho_R} v_R}{\sqrt{\rho_L} + \sqrt{\rho_R}} \\
\hat{h}_t &= \frac{\sqrt{\rho_L} h_{tL} + \sqrt{\rho_R} h_{tR}}{\sqrt{\rho_L} + \sqrt{\rho_R}} \\
\hat{a} &= \left (
(\gamma -1) (\hat{h}_t - \frac{1}{2} (\hat{u}^2 + \hat{v}^2)
\right )^{1/2}
\end{align}
$$

또한 계수 $\hat{\alpha}_i$ 는 다음과 같다.


$$
\begin{align}
\hat{\alpha}_1 &= \Delta \rho - \frac{\Delta p}{\hat{a}^2}\\
\hat{\alpha}_2 &= \hat{\rho} (n_y \Delta u -n_x \Delta v)\\
\hat{\alpha}_3 &= \frac{1}{2 \hat{a}^2} (\Delta p - \hat{\rho}\hat{a} \Delta U) \\
\hat{\alpha}_4 &= \frac{1}{2 \hat{a}^2} (\Delta p + \hat{\rho}\hat{a} \Delta U) \\
\end{align}
$$

### AUSM+ 기법
AUSM+ 기법은 다음과 같이 확장할 수 있다. 1차원과 마찬가지로 Convective 항과 압력항으로 구분한다.

$$
\mathbf{F} = U_n \mathbf{\Phi} + p \mathbf{N}
$$

여기서 $\mathbf{\Phi} = \begin{bmatrix}\rho & \rho u & \rho v & \rho h_t \end{bmatrix}^T$, $\mathbf{N} = \begin{bmatrix}
0 & n_x & n_y & 0
\end{bmatrix}^T$ 이다.

각 항을 차분하면 다음과 같은 경계에서 수치 Flux를 계산할 수 있다.

$$
\mathbf{H}(u_L, u_{R}) = a_{1/2} \left (
\max(M_{1/2}, 0)  \mathbf{\Phi}_L
+
\min(M_{i+1/2}, 0)  \mathbf{\Phi}_R
\right )
+p_{i+1/2} \mathbf{N}
$$

경계에서의 값은 다음 마하수 기반의 Blending 함수로 계산된다. 

$$
\begin{align}
M_{1/2} &= \mathcal{M}^+(M_L) + \mathcal{M}^-(M_{R}) \\
p_{1/2} &= \mathcal{P}^+(M_L) p_L + \mathcal{P}^-(M_{R}) p_{R}
\end{align}
$$

여기서, Blending 함수는 아래와 같다.

$$
\mathcal{M}^\pm = \begin{cases}
\frac{1}{2} (M \pm |M|) & \textrm{if }|M| \geq 1 \\
\frac{1}{4}(M \pm 1)^2 \pm \beta(M^2 -1)^2 & \textrm{otherwise}
\end{cases},
$$

$$
\mathcal{P}^\pm = \begin{cases}
\frac{1}{2} (1 \pm sign(M)) & \textrm{if }|M| \geq 1 \\
\frac{1}{4}(M \pm 1)^(2\pm M) \pm \alpha M (M^2 -1)^2 & \textrm{otherwise}
\end{cases}
$$

Liou가 추천한 $\alpha=3/16$, $\beta=1/8$ 을 사용한다. 좌, 우 마하수는 다음과 같다.

$$
M_L = \frac{U_{n,L}}{a_{1/2}}, M_{R} = \frac{U_{n,R}}{a_{1/2}}
$$

여기서 경계에서의 음속은 Prantl 조건을 만족하도록 다음과 같이 계산한다.

$$
a_{1/2} = \min(\tilde{a}_L, \tilde{a}_{R})
$$

이때,

$$
\tilde{a}_{L,R} = \frac{a^{*2}_{L,R}}{\max(a^*_{L,R}, |U_n|)},
$$

이고, Critical 음속은 다음과 같다.

$$
a^{*2} = \sqrt{2 \frac{\gamma-1}{\gamma+1} h_t }.
$$

### HLLC 기법
HLLC 기법은 다음과 같은 형태로 생각할 수 있다.

$$
\mathbf{H}(\mathbf{U}_L, \mathbf{U}_R, \mathbf{n}) = 
\begin{cases}
\mathbf{F}_L & \textrm{if } 0 < S_L\\
\mathbf{F}_L + S_L (\mathbf{U}_{*L} - \mathbf{U}_L) & \textrm{if } S_L < 0 < S_* \\
\mathbf{F}_R + S_R (\mathbf{U}_{*R} - \mathbf{U}_R) & \textrm{if } S_* < 0 < S_R \\
\mathbf{F}_R & \textrm{if } S_R > 0
\end{cases}
$$

여기서 $()_*$ 는 Intermediate state 이며, Wave speed $S_*$ 는 다음과 같다.

$$
S_* = \frac{p_R - p_L + \rho_L U_{n,L} (S_L - U_{n,L}) - \rho_R U_{n,R} (S_R - U_{n,R})}
{\rho_L (S_L - U_{n,L}) - \rho_R (S_R - U_{n,R})}
$$

좌우 중간 상태 vector는 다음과 같다.

$$
\mathbf{U}_{*L} = 
\frac{S_L - U_{n,L}}{S_L - S_*}
\begin{bmatrix}
\rho_L \\
\rho_L u_L + \rho_L(S_*-U_{n,L})n_x \\
\rho_L v_L + \rho_L(S_*-U_{n,L})n_y \\
\rho_L e_{tL} + (S_* - U_{n,L}) \left (S_* + \frac{p_L}{S_L - U_{n,L}} \right ) 
\end{bmatrix}
$$

$$
\mathbf{U}_{*R} = 
\frac{S_R - U_{n,R}}{S_R - S_*}
\begin{bmatrix}
\rho_R \\
\rho_R u_R + \rho_R (S_*-U_{n,R})n_x \\
\rho_R v_R + \rho_R (S_*-U_{n,R})n_y \\
\rho_R e_{tR} + (S_* - U_{n,R}) \left (S_* + \frac{p_R}{S_L - U_{n,R}} \right ) 
\end{bmatrix},
$$

좌우 에서 Wave speed는 Roe averaged 값으로 계산한다.

$$
S_L = \min(U_{n,L} -a_L, \hat{U}_n -\hat{a} ),
S_R = \max(U_{n,R} +a_R, \hat{U}_n +\hat{a} ),
$$

#### 실습
- Roe's FDS, AUSM+ 기법을 구현하시오
- (Optional) HLLC 기법을 구현하시오

In [3]:
%%writefile solvers/rsolvers/roe.py
from nbutils import array
from solvers.fluid import to_flux

import numba as nb
import numpy as np


def make_roe(gamma=1.4, nvars=4):
    """
    Roe's FDS 계산 함수 생성
    
    Parameters
    ----------
    gamma : float
        Specific heat ratio (기본값: 1.4)
    nvars : integer
        벡터 크기 (기본값: 4)
        
    Return
    ------
    _flux : function
        Roe flux 함수
    """
    ndims = nvars - 2

    @nb.jit(nopython=True)
    def _flux(ul, ur, nf, fn):
        fl, fr = array((nvars,)), array((nvars,))
        vl, vr = array((ndims,)), array((ndims,))
        dv, va = array((ndims,)), array((ndims,))
        ev = array((3, nvars))

        pl, contravl = to_flux(ul, fl, nf)
        pr, contravr = to_flux(ur, fr, nf)

        # Specific enthalpy, contra velocity for left / right
        for jdx in range(ndims):
            vl[jdx] = ul[jdx+1] / ul[0]
            vr[jdx] = ur[jdx+1] / ur[0]

        hl = (ul[nvars-1] + pl)/ul[0]
        hr = (ur[nvars-1] + pr)/ur[0]

        # Difference between two state
        drho = ur[0] - ul[0]
        dp = pr - pl
        dcontrav = contravr - contravl
        for jdx in range(ndims):
            dv[jdx] = vr[jdx] - vl[jdx]

        # Compute Roe averaged density and enthalpy
        rrr = np.sqrt(ur[0]/ul[0])
        ratl = 1.0/(1.0 + rrr)
        ratr = rrr*ratl
        ra = rrr*ul[0]
        ha = hl*ratl + hr*ratr

        for jdx in range(ndims):
            va[jdx] = vl[jdx]*ratl + vr[jdx]*ratr

        contrava = va[0]*nf[0] + va[1]*nf[1]

        qq = 0.5*(va[0]**2 + va[1]**2)
        aasq = (gamma - 1)*(ha - qq)
        aa = np.sqrt(aasq)
        inv_aasq = 1/aasq

        l1 = np.abs(contrava - aa)
        l2 = np.abs(contrava)
        l3 = np.abs(contrava + aa)

        alp1 = 0.5*(dp - ra*aa*dcontrav)*inv_aasq
        alp2 = drho - dp*inv_aasq
        alp3 = 0.5*(dp + ra*aa*dcontrav)*inv_aasq
        
        ev[0, 0] = alp1
        ev[0, nvars-1] = alp1*(ha - aa*contrava)

        ev[1, 0] = alp2
        ev[1, nvars-1] = alp2*qq + ra*(va[0]*dv[0] + va[1]*dv[1] - contrava*dcontrav)

        ev[2, 0] = alp3
        ev[2, nvars-1] = alp3*(ha + aa*contrava)

        for jdx in range(ndims):
            ev[0, 1+jdx] = alp1*(va[jdx] - aa*nf[jdx])
            ev[1, 1+jdx] = alp2*va[jdx] + ra*(dv[jdx] - nf[jdx]*dcontrav)
            ev[2, 1+jdx] = alp3*(va[jdx] + aa*nf[jdx])

        for jdx in range(nvars):
            fn[jdx] = 0.5*(fl[jdx] + fr[jdx]) - 0.5*(l1*ev[0, jdx] + l2*ev[1, jdx] + l3*ev[2, jdx])

    return _flux

Overwriting solvers/rsolvers/roe.py


### 경계 조건 일반화
- 경계조건 커널 재구성
    - `bctypes` 정보에 따라 경계조건 커널을 구성하도록 `make_bc` 커널 생성자 작성
    - 경계 조건에 따른 함수를 호출할 수 있도록 `solvers.bcs` 내 `__init__.py` 파일 구성

In [4]:
%%writefile solvers/bc.py
from nbutils import array
from solvers.bcs import get_bc

import numba as nb
import numpy as np


def make_bc(nfx, nfy, si, sj, bctypes, nvars=4, npads=(1,1)):
    """
    BC kernel genertor

    Parameter
    ---------
    nfx - integer
        xi 방향 격자 개수
    nfy - integer
        eta 방향 격자 개수
    si : array
        xi 방향 Face metric
    sj : array
        eta 방향 Face metric
    bctypes : dict
        경계조건 Dictionary
    nvars : integer
        벡터 크기 (기본값: 4)
    npads : integer
        경계를 위한 padding 개수 (기본값: (1,1))
    """
    # Compute unit normal vector
    ni = si / np.linalg.norm(si, axis=0)
    nj = sj / np.linalg.norm(sj, axis=0)    
    
    # Generate bc sweep kernel
    if bctypes['imin'][0] == 'periodic':
        bc_imin = make_bc_i_periodic(nfx+npads[0]-1, npads[0]-1, nfy, -1, nvars, npads)
        bc_imax = make_bc_i_periodic(npads[0], nfx+npads[0], nfy, +1, nvars, npads)
    else:
        bc_imin = make_bc_i(npads[0], npads[0]-1, nfy, bctypes['imin'], ni, nvars, npads)
        bc_imax = make_bc_i(nfx+npads[0]-1, nfx+npads[0], nfy, bctypes['imax'], ni, nvars, npads)
        
    if bctypes['jmin'][0] == 'periodic':
        bc_jmin = make_bc_j_periodic(nfy+npads[1] - 1, npads[1]-1, nfx, -1, nvars, npads)
        bc_jmax = make_bc_j_periodic(npads[1], nfy+npads[1], nfx, +1, nvars, npads)
    else:
        bc_jmin = make_bc_j(npads[1], npads[1]-1, nfx, bctypes['jmin'], nj, nvars, npads)
        bc_jmax = make_bc_j(nfy+npads[1]-1, nfy+npads[1], nfx, bctypes['jmax'], nj, nvars, npads)
    
    @nb.jit(nopython=True)
    def _run(u):
        bc_imin(u)
        bc_imax(u)
        bc_jmin(u)
        bc_jmax(u)
        
    return _run


def make_bc_i(i, ip, nfy, bcinfo, nf, nvars, npads=(1,1)):    
    """
    i 방향 BC Sweep 커널 생성

    Parameter
    ---------
    i - integer
        Innder index
    ip - integer
        Ghost cell index
    nfy - integer
        eta 방향 격자 개수
    bcinfo - turple
        경계조건 이름과 조건 (dict)
    nf - array
        수직벡터
    nvars : integer
        벡터 크기 (기본값: 4)
    npads : integer
        경계를 위한 padding 개수 (기본값: (1,1))
    """
    bcf = get_bc(bcinfo[0], nvars=nvars, **bcinfo[1])
        
    @nb.jit(nopython=True)
    def _run(u):               
        ur = array((nvars,))
        
        for j in range(npads[1], npads[1]+nfy):
            bcf(u[:, j, i], ur, nf[:, j-npads[1], i-npads[0]])
            for k in range(nvars):
                u[k, j, ip] = ur[k]
                
    return _run
                
    
def make_bc_j(j, jp, nfx, bcinfo, nf, nvars, npads=(1,1)):
    """
    j 방향 BC Sweep 커널 생성

    Parameter
    ---------
    j - integer
        Innder index
    jp - integer
        Ghost cell index
    nfx - integer
        xi 방향 격자 개수
    bcinfo - turple
        경계조건 이름과 조건 (dict)
    nf - array
        수직벡터
    nvars : integer
        벡터 크기 (기본값: 4)
    npads : integer
        경계를 위한 padding 개수 (기본값: (1,1))
    """
    bcf = get_bc(bcinfo[0], nvars=nvars, **bcinfo[1])
    
    @nb.jit(nopython=True)
    def _run(u):
        ur = array((nvars,))
        
        for i in range(npads[0], npads[0]+nfx):
            bcf(u[:, j, i], ur, nf[:, j-npads[1], i-npads[0]])
            for k in range(nvars):
                u[k, jp, i] = ur[k]
                    
    return _run


def make_bc_i_periodic(i, ip, nfy, sign, nvars, npads=(1,1)):
    """
    i 방향 대칭 BC 커널 생성

    Parameter
    ---------
    i - integer
        Innder index
    ip - integer
        Ghost cell index
    nfy - integer
        eta 방향 격자 개수
    sign - integer
        방향 (-1: min, +1: max)
    nvars : integer
        벡터 크기 (기본값: 4)
    npad : integer
        i 방향 경계를 위한 padding 개수 (기본값: 1)
    """
    @nb.jit(nopython=True)
    def _run(u):
        for k in range(nvars):
            for j in range(npads[1], npads[1]+nfy):
                for ix in range(npads[0]):
                    u[k, j, ip + sign*ix] = u[k, j, i+ sign*ix]

    return _run


def make_bc_j_periodic(j, jp, nfx, sign, nvars, npads=(1,1)):
    """
    j 방향 대칭 BC 커널 생성

    Parameter
    ---------
    j - integer
        Innder index
    jp - integer
        Ghost cell index
    nfx - integer
        xi 방향 격자 개수
    sign - integer
        방향 (-1: min, +1: max)
    nvars : integer
        벡터 크기 (기본값: 4)
    npad : integer
        j 방향 경계를 위한 padding 개수 (기본값: 1)
    """
    @nb.jit(nopython=True)
    def _run(u):
        for k in range(nvars):
            for jx in range(npads[1]):
                for i in range(npads[0], npads[0]+nfx):                
                    u[k, jp + sign*jx, i] = u[k, j+ sign*jx, i ]
                
    return _run

Overwriting solvers/bc.py


In [5]:
%%writefile solvers/bcs/__init__.py

from solvers.bcs.far import make_bc_far
from solvers.bcs.wall import make_bc_wall
from solvers.bcs.sup import make_bc_sup_in, make_bc_sup_out
from solvers.bcs.zero import make_bc_zero


def get_bc(name, nvars, **kwargs):
    bc = eval('make_bc_'+name)(nvars, **kwargs)
    return bc

Overwriting solvers/bcs/__init__.py


## 시간차분 기법
- 수치 진동의 발달을 억제하기 위해 3차 정확도 TVD Runge Kutta (SSP RK) 기법을 사용한다.

- TVD Runge Kutta scheme

$$
\begin{align}
u^{*}_j &= u^n_j + \Delta t RHS (u^n)  \\
u^{*}_j &= \frac{3}{4} u^n_j + \frac{1}{4} u^{*}_j + \frac{1}{4} \Delta t RHS (u^*) \\
u^{n+1}_j &= \frac{1}{3} u^n_j + \frac{2}{3} u^{*}_j + \frac{2}{3} \Delta t RHS(u^*)
\end{align}
$$


### 실습

In [6]:
%%writefile solvers/unsteady/tvdrk3.py
import numpy as np


def make_tvdrk3(nfx, nfy, rhside, nvars=4, npads=(1,1)):
    """
    TVD Runge Kutta 3rd 계산 커널 생성
    
    Parameters
    ----------
    nfx - integer
        xi 방향 격자 개수
    nfy - integer
        eta 방향 격자 개수
    rhside : kernel
        우변 계산 커널
    nvars : integer
        벡터 크기 (기본값: 3)
    npad : integer
        Solution 벡터의 Padding 개수
    """
    # Temporal arrays
    du = np.zeros((nvars, nfy+2*npads[1], nfx+2*npads[0]))
    u0 = np.zeros((nvars, nfy+2*npads[1], nfx+2*npads[0]))
    
    # Temporal arrays for flux
    ff = np.zeros((nvars, nfy, nfx+1))
    fg = np.zeros((nvars, nfy+1, nfx))
    
    def _step(dt, u):
        # TVD-RK3
        rhside(u, du, ff, fg)
        u0[:] = u
        u += dt*du

        rhside(u, du, ff, fg)
        u[:] = (3*u0 + u +dt*du)/4

        rhside(u, du, ff, fg)
        u[:] = (u0 + 2*u + 2*dt*du)/3
        
    return _step

Overwriting solvers/unsteady/tvdrk3.py


## 실행
- 격자 생성
- 계산 벡터 생성 : $U$
- Kernels 생성
- 초기화
- Main Loop

In [7]:
from matplotlib import pylab as plt

# 모듈 로드
from solvers.plot3d import read_plot3d_b, write_q
from solvers.grid import cell_jacobian, face_metric, cell_x
from solvers.fluid import to_conservative, to_primitive
from solvers.rhside import make_rhside
from solvers.unsteady.timestep import make_timestep
from solvers.unsteady.tvdrk3 import make_tvdrk3

import numpy as np

plt.style.use('ggplot')
plt.rcParams['figure.dpi'] = 150

In [8]:
# Constants
gamma = 1.4
nvars = 4
npads = (1, 2)

cfl = 0.9

# Schemes
flux = 'rusanov'
limiter = 'vanleer'

# Initial conditions (Primitive)
qpl = (1.0, 0.75, 0.0, 1.0)
qpr = (0.125, 0.0, 0.0, 0.1)
xbk = -0.2
target_time = 0.2

# BC conditions
bctypes = {
    'imin' : ('zero', {}), 
    'imax' : ('zero', {}),
    'jmin' : ('periodic', {}), 
    'jmax' : ('periodic', {}),
}

# Plot3D 격자 읽기
raw = read_plot3d_b('grids/tube.xyz')

# Parse point
x, y = raw[:2, 0]

# Compute area and metric
jac = cell_jacobian(x ,y)
si, sj = face_metric(x, y)

# Cell Center
xc, yc =cell_x(x, y)

# Array
jmx, imx = x.shape
jmx -= 1
imx -= 1
q = np.empty((nvars, jmx+2*npads[1], imx+2*npads[0]))
            
# Generate kernels
rhside, bc = make_rhside(imx, jmx, jac, si, sj, flux, bctypes, limiter, gamma, nvars, npads)
timestep = make_timestep(imx, jmx, jac, si, sj, cfl, gamma, nvars, npads)
step = make_tvdrk3(imx, jmx, rhside, nvars, npads)

# Initialize
ql = np.empty(nvars)
qr = np.empty(nvars)

to_conservative(qpl, ql)
to_conservative(qpr, qr)

for j in range(jmx):
    for i in range(imx):
        if xc[j,i] < xbk:
            q[:, j+npads[1], i+npads[0]] = ql
        else:
            q[:, j+npads[1], i+npads[0]] = qr

# Main Loop        
t = 0
n = 0
while abs(t - target_time) > np.finfo(1.0).eps:
    # Compute time step
    dt = min(timestep(q), target_time - t)
    
    # Step
    step(dt, q)
    t += dt
    n += 1

print("{} iterations until t={:.3f}".format(n, t))

Riemann Solver : rusanov
Van Leer limiter
184 iterations until t=0.200


In [9]:
# Write solution
bc(q)

# Adjust solution array considering padding
_, jqy, iqx = q.shape
ie = (0, iqx)
je = (0, jqy)

if npads[0] > 1:
    ie = npads[0] - 1, -(npads[0] - 1)
    
if npads[1] > 1:
    je = npads[1] - 1, -(npads[1] - 1)

write_q('sol.q', q[:, je[0]:je[1], ie[0]:ie[1]])

## 가시화
- Paraview 또는 Tecplot으로 유동을 확인하다.

    ```bash
    user@localhost: ~$ paraview grids/tube.xyz
    ```

    - `sol.q` 파일을 Q File Name으로 설정한다.